# Initialization

In [1]:
# Laborers
import os
import pandas as pd
import numpy as np

# Image workers
import cv2 as cv
from PIL import Image, ImageOps

# CNN
## CNN infrastructure
import torch
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

## CNN image workers
from torchvision import transforms, datasets
from torchvision.transforms import v2
device = torch.device(torch.accelerator.current_accelerator().type)

# Main

## Image processing

In [2]:
crop_coords = [
    (85, 340, 234, 560),
    (85, 610, 234, 830),
    (85, 870, 234, 1090),
    (85, 1140, 234, 1360),
    (85, 1400, 234, 1620),
    (85, 1670, 234, 1890)
]

In [ ]:
results = []

# Crop and save.
i = 0
for result in os.listdir(r"cq_results"):
    # Summon and crop.
    img = cv.imread(f"cq_results/{result}")
    # Find average color in cropped area.
    for coord in crop_coords:
        crop = img[coord[1]:coord[3],
                   coord[0]:coord[2]]
        cv.imwrite(f"./gcq_training_images/ghost/{i:03}.png", crop)
        i += 1

## CNN

In [4]:
# Set transformer.
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

# Load and batch training data.
train = datasets.ImageFolder(root = "training/", 
                             transform = transform)
train_load = DataLoader(train, 
                        batch_size = 1, 
                        shuffle = True)

# Attach label (key) to index (value).
id = {label: index for label, index in 
      enumerate(sorted(os.listdir("training/")))}

In [5]:
# Basic neural network.
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(kernel_size = 2, stride = 2)
        self.fc1 = nn.Linear(46656, 10) 

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        return x

# Activate.
net = Net().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), lr=0.001)

In [6]:
for epoch in range(2):
    running_loss = 0.0
    for i, data in enumerate(train_load, 0):
        # Designate inputs and labels from training.
        inputs, labels = data[0].to(device), data[1].to(device)
        # Nullify parameter gradients.
        optimizer.zero_grad()

        # Iterate.
        outputs = net(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

# Save.
torch.save(net.state_dict(),
           "gcq.pth")

## Emergency Test Validation thing

In [7]:
#results = []
#
## Load and batch new data.
#test = datasets.ImageFolder(root = "base_images/", 
#                             transform = transform)
#test_load = DataLoader(test, 
#                       batch_size = 1)
#
## Process new entries.
#for images, labels in test_load:
#    pred = net(images.to(device))
#    _, predicted = torch.max(pred.data, 1)
#    results.append(predicted[0].item())